# 03 · Results Extraction (R)

Converts network estimation outputs from `02_ggm_estimation` into analysis-ready
tables and KG-ready metadata tables for downstream knowledge-graph encoding.

## Inputs  (`../results/networks/`)

| File | Source |
|------|--------|
| `networks_summary.csv` | Written by Cell 5 of `02_ggm_estimation` |
| `edges_wave{W}_trust{S}_n{N}_{spec}.csv` | Edge lists per context × spec |
| `adj_wave{W}_trust{S}_n{N}_{spec}.csv` | Adjacency matrices per context × spec |
| `boot_edges_wave{W}_trust{S}_n{N}_{spec}.rds` | Bootstrap objects (g05 + g075 only) |

## Specifications

| Label | γ | Thresholded | In BKG |
|-------|---|-------------|--------|
| `g05`     | 0.50 | No  | Yes |
| `g075`    | 0.75 | No  | Yes |
| `g05_thr` | 0.50 | Yes | No  |

All three specs appear in analysis tables.  Only `g05` and `g075` are encoded in
the BKG (12 instances = 3 waves × 2 strata × 2 specs).

## Outputs  (`../results/networks/`)

**Analysis tables (all specs)**
- `edges_all.csv` — all nonzero undirected edges with context/spec metadata
- `bridge_edges.csv` — belief/affect ↔ behavior edges
- `node_metrics.csv` — degree and strength per node per context/spec
- `context_summary.csv` — context-level edge counts and behavior connectivity
- `edge_differences_high_minus_low.csv` — edge-wise (high − low) differences per wave/spec
- `bridge_differences_high_minus_low.csv` — bridge subset of the above
- `focal_belief_bridges.csv` — SEVERITY ↔ behavior edges
- `top_edges_top10.csv` — top-10 |weight| edges per network instance
- `edge_bootstrap_metrics.csv` — bootstrap CI/stability (g05 + g075 only)
- `edge_metrics.csv` — point estimates merged with bootstrap summaries

**KG-ready metadata tables (g05 + g075 only)**
- `kg_contexts.csv`
- `kg_gamma_specs.csv`
- `kg_network_instances.csv`

In [1]:
# ── Packages ──────────────────────────────────────────────────────────────────
suppressPackageStartupMessages({
  library(dplyr)
  library(readr)
  library(stringr)
  library(tidyr)
  library(purrr)
  library(bootnet)   # needed in Cell 13 for readRDS of bootnet objects
})

NETWORK_DIR <- file.path("..", "results", "networks")

In [2]:
# ── Node metadata — defined as constants, NOT inferred from files ─────────────
# NODE_VARS order must match the estimation notebook exactly.
# Defining it here as a constant (rather than reading from the first adj file)
# makes this notebook independent of which file happens to be listed first.
NODE_VARS <- c(
  "SEVERITY",
  "AFF_FEAR", "AFF_WORRY", "WORRY_HEALTH_SYSTEM",
  "USE2_MASK", "USE2_SPACE150", "USE2_HANDWASH20", "USE2_AVOID"
)

NODE_DOMAIN <- c(
  SEVERITY            = "belief",
  AFF_FEAR            = "affect",
  AFF_WORRY           = "affect",
  WORRY_HEALTH_SYSTEM = "affect",
  USE2_MASK           = "behavior",
  USE2_SPACE150       = "behavior",
  USE2_HANDWASH20     = "behavior",
  USE2_AVOID          = "behavior"
)

FOCAL_NODE <- "SEVERITY"
PSY_NODES  <- names(NODE_DOMAIN)[NODE_DOMAIN %in% c("belief", "affect")]
BEH_NODES  <- names(NODE_DOMAIN)[NODE_DOMAIN == "behavior"]

# NODE_ROLE distinguishes the focal belief from all other nodes.
# Domain membership (belief / affect / behavior) is captured by NODE_DOMAIN.
NODE_ROLE <- setNames(
  ifelse(names(NODE_DOMAIN) == FOCAL_NODE, "focal_belief", "non_focal"),
  names(NODE_DOMAIN)
)

# ── Specification lookup ────────────────────────────────────────────────────────
# in_kg: whether the spec is stored as a BKG NetworkInstance.
# Only the main specification (g05, Pearson) is stored in the BKG.
# All others are robustness checks reported in the Results chapter only.
GAMMA_LOOKUP <- tibble(
  spec        = c("g05",      "g075",      "g05_thr",      "g05_spearman"),
  spec_id     = c("gamma_05", "gamma_075", "gamma_05_thr", "gamma_05_spearman"),
  gamma_ebic  = c(0.5,         0.75,        0.5,            0.5),
  thresholded = c(FALSE,       FALSE,       TRUE,           FALSE),
  cor_method  = c("pearson",   "pearson",   "pearson",      "spearman"),
  in_kg       = c(TRUE,        FALSE,       FALSE,          FALSE)
)

KG_SPECS <- filter(GAMMA_LOOKUP, in_kg)

TRUST_VARIABLE          <- "TRUST_RKI"
TRUST_GROUPING_RULE     <- "within-wave terciles"
MIDDLE_TERCILE_EXCLUDED <- TRUE

cat("NODE_VARS  :", paste(NODE_VARS,  collapse = ", "), "\n")
cat("KG specs   :", paste(KG_SPECS$spec, collapse = ", "), "\n")

NODE_VARS  : SEVERITY, AFF_FEAR, AFF_WORRY, WORRY_HEALTH_SYSTEM, USE2_MASK, USE2_SPACE150, USE2_HANDWASH20, USE2_AVOID 
KG specs   : g05 


In [3]:
# ── Load networks_summary.csv and discover files ──────────────────────────────
# networks_summary.csv is the authoritative record of what was estimated.
# File discovery is then cross-validated against it.

SUMMARY_PATH <- file.path(NETWORK_DIR, "networks_summary.csv")
if (!file.exists(SUMMARY_PATH))
  stop("networks_summary.csv not found. Run 02_ggm_estimation first.")

net_summary <- read_csv(SUMMARY_PATH, show_col_types = FALSE)
required_summary_cols <- c("wave","trust_stratum","n_cc","spec","gamma_ebic","threshold_applied")
missing_summary_cols  <- setdiff(required_summary_cols, names(net_summary))
if (length(missing_summary_cols) > 0)
  stop("networks_summary.csv missing columns: ", paste(missing_summary_cols, collapse=", "))

cat("networks_summary.csv: ", nrow(net_summary), " rows\n")

# ── Regex patterns: g05_thr ordered before g05 to prevent partial match ────────
SPEC_PATTERN <- paste(
  GAMMA_LOOKUP$spec[order(-nchar(GAMMA_LOOKUP$spec))],
  collapse = "|"
)

parse_meta <- function(path) {
  fname   <- basename(path)
  pattern <- paste0(
    "^(edges|adj|boot_edges)_wave(\\d+)_trust(low|high)_n(\\d+)_(",
    SPEC_PATTERN, ")\\.(csv|rds)$"
  )
  m <- str_match(fname, pattern)
  if (any(is.na(m))) return(NULL)
  tibble(
    kind          = m[2],
    wave          = as.integer(m[3]),
    trust_stratum = m[4],
    n_cc          = as.integer(m[5]),
    spec          = m[6],
    fname         = fname,
    path          = path
  )
}

edge_files <- list.files(NETWORK_DIR, pattern = "\\.csv$", full.names = TRUE)
edge_files <- edge_files[grepl("^edges_wave", basename(edge_files))]
adj_files  <- list.files(NETWORK_DIR, pattern = "\\.csv$", full.names = TRUE)
adj_files  <- adj_files[grepl("^adj_wave",   basename(adj_files))]
boot_files <- list.files(NETWORK_DIR, pattern = "\\.rds$", full.names = TRUE)
boot_files <- boot_files[grepl("^boot_edges_wave", basename(boot_files))]

edge_meta <- map_dfr(edge_files, parse_meta)
adj_meta  <- map_dfr(adj_files,  parse_meta)
boot_meta <- if (length(boot_files) > 0) map_dfr(boot_files, parse_meta) else tibble()

if (nrow(edge_meta) == 0) stop("No edge CSVs found in ", NETWORK_DIR)
if (nrow(adj_meta)  == 0) stop("No adjacency CSVs found in ", NETWORK_DIR)

stopifnot(all(edge_meta$spec %in% GAMMA_LOOKUP$spec))
stopifnot(all(adj_meta$spec  %in% GAMMA_LOOKUP$spec))
if (nrow(boot_meta) > 0)
  stopifnot(all(boot_meta$spec %in% GAMMA_LOOKUP$spec))

# ── Cross-validate file-parsed metadata against networks_summary.csv ───────────
# Every (wave, trust_stratum, n_cc, spec) row in the edge metadata should
# appear in networks_summary.csv; a mismatch indicates a stale file or
# a pipeline inconsistency.
edge_keys <- edge_meta %>% distinct(wave, trust_stratum, n_cc, spec)
summary_keys <- net_summary %>% distinct(wave, trust_stratum, n_cc, spec)
orphan_edges <- anti_join(edge_keys, summary_keys,
                          by = c("wave","trust_stratum","n_cc","spec"))
if (nrow(orphan_edges) > 0) {
  warning("Edge files found with no matching row in networks_summary.csv:\n",
          paste(capture.output(print(orphan_edges)), collapse="\n"))
}

# ── Verify adjacency files cover the same set as edge files ───────────────────
adj_keys  <- adj_meta  %>% distinct(wave, trust_stratum, n_cc, spec)
edge_no_adj <- anti_join(edge_keys, adj_keys, by = c("wave","trust_stratum","n_cc","spec"))
adj_no_edge <- anti_join(adj_keys, edge_keys, by = c("wave","trust_stratum","n_cc","spec"))
if (nrow(edge_no_adj) > 0) warning("Edge files without matching adj: ", nrow(edge_no_adj))
if (nrow(adj_no_edge) > 0) warning("Adj files without matching edge: ", nrow(adj_no_edge))

# ── Verify node order in adjacency files matches NODE_VARS ────────────────────
# Check first file; failures here indicate a node-ordering mismatch between
# the estimation and extraction notebooks.
W0m <- as.matrix(read.csv(adj_meta$path[1], row.names = 1, check.names = FALSE))
if (!identical(rownames(W0m), NODE_VARS))
  stop(
    "Node order in adjacency file does not match NODE_VARS constant.\n",
    "  File nodes : ", paste(rownames(W0m),   collapse=", "), "\n",
    "  NODE_VARS  : ", paste(NODE_VARS,        collapse=", ")
  )

cat("Found edge files    :", nrow(edge_meta), "\n")
cat("Found adj files     :", nrow(adj_meta),  "\n")
cat("Found bootstrap RDS :", nrow(boot_meta), "\n")
cat("  (bootstrap present for g05 and g075; g05_thr has no bootstrap by design)\n")
cat("Node order verified :", paste(NODE_VARS, collapse=", "), "\n")

networks_summary.csv:  24  rows
Found edge files    : 24 
Found adj files     : 24 
Found bootstrap RDS : 12 
  (bootstrap present for g05 and g075; g05_thr has no bootstrap by design)
Node order verified : SEVERITY, AFF_FEAR, AFF_WORRY, WORRY_HEALTH_SYSTEM, USE2_MASK, USE2_SPACE150, USE2_HANDWASH20, USE2_AVOID 


In [4]:
# ── Load all nonzero edges into one tidy table ────────────────────────────────
# edge_a / edge_b: canonical ordering (pmin / pmax) so every undirected edge
# has a single consistent representation regardless of which direction the
# adjacency matrix stored it.
# edge_id format: {network_id}__{edge_a}__{edge_b}
# The double-underscore separator works because NODE_VARS names contain
# no underscores that could create ambiguity.

edges_all <- edge_meta %>%
  mutate(dat = map(path, ~ read_csv(.x, show_col_types = FALSE))) %>%
  select(wave, trust_stratum, n_cc, spec, dat) %>%
  unnest(dat) %>%
  filter(weight != 0) %>%
  left_join(GAMMA_LOOKUP, by = "spec") %>%
  mutate(
    context_id  = paste0("ctx_w",  wave, "_", trust_stratum),
    network_id  = paste0("net_", spec, "_w", wave, "_", trust_stratum),
    edge_a      = pmin(from, to),
    edge_b      = pmax(from, to),
    edge_id     = paste(network_id, edge_a, edge_b, sep = "__"),
    abs_weight  = abs(weight),
    from_domain = NODE_DOMAIN[from],
    to_domain   = NODE_DOMAIN[to],
    from_role   = NODE_ROLE[from],
    to_role     = NODE_ROLE[to]
  ) %>%
  arrange(spec, wave, trust_stratum, desc(abs_weight))

# Sanity: no duplicate edge_ids within a network
dup_ids <- edges_all %>%
  group_by(edge_id) %>%
  filter(n() > 1) %>%
  nrow()
if (dup_ids > 0)
  stop(dup_ids, " duplicate edge_id values detected. Check adjacency export.")

write_csv(edges_all, file.path(NETWORK_DIR, "edges_all.csv"))
cat("Saved: edges_all.csv —", nrow(edges_all), "rows\n")
head(edges_all, 10)

Saved: edges_all.csv — 442 rows


wave,trust_stratum,n_cc,spec,from,to,weight,spec_id,gamma_ebic,thresholded,⋯,context_id,network_id,edge_a,edge_b,edge_id,abs_weight,from_domain,to_domain,from_role,to_role
<int>,<chr>,<int>,<chr>,<chr>,<chr>,<dbl>,<chr>,<dbl>,<lgl>,⋯,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>
55,high,309,g05,AFF_FEAR,AFF_WORRY,0.5773954,gamma_05,0.5,FALSE,⋯,ctx_w55_high,net_g05_w55_high,AFF_FEAR,AFF_WORRY,net_g05_w55_high__AFF_FEAR__AFF_WORRY,0.5773954,affect,affect,non_focal,non_focal
55,high,309,g05,USE2_AVOID,USE2_SPACE150,0.2379412,gamma_05,0.5,FALSE,⋯,ctx_w55_high,net_g05_w55_high,USE2_AVOID,USE2_SPACE150,net_g05_w55_high__USE2_AVOID__USE2_SPACE150,0.2379412,behavior,behavior,non_focal,non_focal
55,high,309,g05,USE2_MASK,USE2_SPACE150,0.2135914,gamma_05,0.5,FALSE,⋯,ctx_w55_high,net_g05_w55_high,USE2_MASK,USE2_SPACE150,net_g05_w55_high__USE2_MASK__USE2_SPACE150,0.2135914,behavior,behavior,non_focal,non_focal
55,high,309,g05,USE2_HANDWASH20,USE2_SPACE150,0.2033636,gamma_05,0.5,FALSE,⋯,ctx_w55_high,net_g05_w55_high,USE2_HANDWASH20,USE2_SPACE150,net_g05_w55_high__USE2_HANDWASH20__USE2_SPACE150,0.2033636,behavior,behavior,non_focal,non_focal
55,high,309,g05,AFF_FEAR,SEVERITY,0.1963276,gamma_05,0.5,FALSE,⋯,ctx_w55_high,net_g05_w55_high,AFF_FEAR,SEVERITY,net_g05_w55_high__AFF_FEAR__SEVERITY,0.1963276,affect,belief,non_focal,focal_belief
55,high,309,g05,USE2_HANDWASH20,USE2_MASK,0.1832615,gamma_05,0.5,FALSE,⋯,ctx_w55_high,net_g05_w55_high,USE2_HANDWASH20,USE2_MASK,net_g05_w55_high__USE2_HANDWASH20__USE2_MASK,0.1832615,behavior,behavior,non_focal,non_focal
55,high,309,g05,AFF_FEAR,WORRY_HEALTH_SYSTEM,0.1779615,gamma_05,0.5,FALSE,⋯,ctx_w55_high,net_g05_w55_high,AFF_FEAR,WORRY_HEALTH_SYSTEM,net_g05_w55_high__AFF_FEAR__WORRY_HEALTH_SYSTEM,0.1779615,affect,affect,non_focal,non_focal
55,high,309,g05,USE2_AVOID,USE2_HANDWASH20,0.1445860,gamma_05,0.5,FALSE,⋯,ctx_w55_high,net_g05_w55_high,USE2_AVOID,USE2_HANDWASH20,net_g05_w55_high__USE2_AVOID__USE2_HANDWASH20,0.1445860,behavior,behavior,non_focal,non_focal
55,high,309,g05,SEVERITY,USE2_AVOID,0.1407643,gamma_05,0.5,FALSE,⋯,ctx_w55_high,net_g05_w55_high,SEVERITY,USE2_AVOID,net_g05_w55_high__SEVERITY__USE2_AVOID,0.1407643,belief,behavior,focal_belief,non_focal


In [5]:
# ── Bridge edges: belief/affect ↔ behavior ────────────────────────────────────
bridge_df <- edges_all %>%
  filter(
    (from_domain %in% c("belief", "affect") & to_domain == "behavior") |
    (from_domain == "behavior"              & to_domain %in% c("belief", "affect"))
  ) %>%
  arrange(spec, wave, trust_stratum, desc(abs_weight))

write_csv(bridge_df, file.path(NETWORK_DIR, "bridge_edges.csv"))
cat("Saved: bridge_edges.csv —", nrow(bridge_df), "rows\n")
head(bridge_df, 10)

Saved: bridge_edges.csv — 181 rows


wave,trust_stratum,n_cc,spec,from,to,weight,spec_id,gamma_ebic,thresholded,⋯,context_id,network_id,edge_a,edge_b,edge_id,abs_weight,from_domain,to_domain,from_role,to_role
<int>,<chr>,<int>,<chr>,<chr>,<chr>,<dbl>,<chr>,<dbl>,<lgl>,⋯,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>
55,high,309,g05,SEVERITY,USE2_AVOID,0.14076434,gamma_05,0.5,FALSE,⋯,ctx_w55_high,net_g05_w55_high,SEVERITY,USE2_AVOID,net_g05_w55_high__SEVERITY__USE2_AVOID,0.14076434,belief,behavior,focal_belief,non_focal
55,high,309,g05,SEVERITY,USE2_SPACE150,0.12741391,gamma_05,0.5,FALSE,⋯,ctx_w55_high,net_g05_w55_high,SEVERITY,USE2_SPACE150,net_g05_w55_high__SEVERITY__USE2_SPACE150,0.12741391,belief,behavior,focal_belief,non_focal
55,high,309,g05,SEVERITY,USE2_MASK,-0.08060195,gamma_05,0.5,FALSE,⋯,ctx_w55_high,net_g05_w55_high,SEVERITY,USE2_MASK,net_g05_w55_high__SEVERITY__USE2_MASK,0.08060195,belief,behavior,focal_belief,non_focal
55,high,309,g05,USE2_MASK,WORRY_HEALTH_SYSTEM,0.07801505,gamma_05,0.5,FALSE,⋯,ctx_w55_high,net_g05_w55_high,USE2_MASK,WORRY_HEALTH_SYSTEM,net_g05_w55_high__USE2_MASK__WORRY_HEALTH_SYSTEM,0.07801505,behavior,affect,non_focal,non_focal
55,high,309,g05,AFF_FEAR,USE2_AVOID,0.07112662,gamma_05,0.5,FALSE,⋯,ctx_w55_high,net_g05_w55_high,AFF_FEAR,USE2_AVOID,net_g05_w55_high__AFF_FEAR__USE2_AVOID,0.07112662,affect,behavior,non_focal,non_focal
55,high,309,g05,SEVERITY,USE2_HANDWASH20,0.03889617,gamma_05,0.5,FALSE,⋯,ctx_w55_high,net_g05_w55_high,SEVERITY,USE2_HANDWASH20,net_g05_w55_high__SEVERITY__USE2_HANDWASH20,0.03889617,belief,behavior,focal_belief,non_focal
55,high,309,g05,AFF_WORRY,USE2_AVOID,0.03485093,gamma_05,0.5,FALSE,⋯,ctx_w55_high,net_g05_w55_high,AFF_WORRY,USE2_AVOID,net_g05_w55_high__AFF_WORRY__USE2_AVOID,0.03485093,affect,behavior,non_focal,non_focal
55,high,309,g05,USE2_AVOID,WORRY_HEALTH_SYSTEM,0.03478318,gamma_05,0.5,FALSE,⋯,ctx_w55_high,net_g05_w55_high,USE2_AVOID,WORRY_HEALTH_SYSTEM,net_g05_w55_high__USE2_AVOID__WORRY_HEALTH_SYSTEM,0.03478318,behavior,affect,non_focal,non_focal
55,high,309,g05,AFF_WORRY,USE2_HANDWASH20,0.03162142,gamma_05,0.5,FALSE,⋯,ctx_w55_high,net_g05_w55_high,AFF_WORRY,USE2_HANDWASH20,net_g05_w55_high__AFF_WORRY__USE2_HANDWASH20,0.03162142,affect,behavior,non_focal,non_focal


In [6]:
# ── Node metrics: degree and strength from adjacency matrices ─────────────────
# Adjacency matrices are read directly (rather than re-deriving from edge lists)
# to stay faithful to the exported estimation output.
# Diagonal is zeroed before computing degree and strength.

node_metrics <- adj_meta %>%
  mutate(
    W = map(path, function(p) {
      M <- as.matrix(read.csv(p, row.names = 1, check.names = FALSE))
      # Enforce canonical node order; stop if nodes are missing or extra
      if (!all(NODE_VARS %in% rownames(M)))
        stop("Adj matrix ", basename(p), " is missing expected nodes.")
      M <- M[NODE_VARS, NODE_VARS, drop = FALSE]
      diag(M) <- 0
      M
    })
  ) %>%
  mutate(
    node_stats = map(W, function(M) {
      tibble(
        node     = NODE_VARS,
        node_id  = NODE_VARS,
        degree   = as.integer(rowSums(M != 0)),
        strength = as.numeric(rowSums(abs(M)))
      )
    })
  ) %>%
  select(spec, wave, trust_stratum, n_cc, node_stats) %>%
  unnest(node_stats) %>%
  left_join(GAMMA_LOOKUP, by = "spec") %>%
  mutate(
    context_id = paste0("ctx_w", wave, "_", trust_stratum),
    network_id = paste0("net_", spec, "_w", wave, "_", trust_stratum),
    domain     = NODE_DOMAIN[node],
    role       = NODE_ROLE[node]
  ) %>%
  arrange(spec, wave, trust_stratum, domain, node)

write_csv(node_metrics, file.path(NETWORK_DIR, "node_metrics.csv"))
cat("Saved: node_metrics.csv —", nrow(node_metrics), "rows\n")
head(node_metrics, 10)

Saved: node_metrics.csv — 192 rows


spec,wave,trust_stratum,n_cc,node,node_id,degree,strength,spec_id,gamma_ebic,thresholded,cor_method,in_kg,context_id,network_id,domain,role
<chr>,<int>,<chr>,<int>,<chr>,<chr>,<int>,<dbl>,<chr>,<dbl>,<lgl>,<chr>,<lgl>,<chr>,<chr>,<chr>,<chr>
g05,55,high,309,AFF_FEAR,AFF_FEAR,6,1.0544797,gamma_05,0.5,FALSE,pearson,TRUE,ctx_w55_high,net_g05_w55_high,affect,non_focal
g05,55,high,309,AFF_WORRY,AFF_WORRY,5,0.7797262,gamma_05,0.5,FALSE,pearson,TRUE,ctx_w55_high,net_g05_w55_high,affect,non_focal
g05,55,high,309,WORRY_HEALTH_SYSTEM,WORRY_HEALTH_SYSTEM,7,0.5169383,gamma_05,0.5,FALSE,pearson,TRUE,ctx_w55_high,net_g05_w55_high,affect,non_focal
g05,55,high,309,USE2_AVOID,USE2_AVOID,7,0.7157924,gamma_05,0.5,FALSE,pearson,TRUE,ctx_w55_high,net_g05_w55_high,behavior,non_focal
g05,55,high,309,USE2_HANDWASH20,USE2_HANDWASH20,7,0.6438941,gamma_05,0.5,FALSE,pearson,TRUE,ctx_w55_high,net_g05_w55_high,behavior,non_focal
g05,55,high,309,USE2_MASK,USE2_MASK,6,0.6262256,gamma_05,0.5,FALSE,pearson,TRUE,ctx_w55_high,net_g05_w55_high,behavior,non_focal
g05,55,high,309,USE2_SPACE150,USE2_SPACE150,5,0.8069789,gamma_05,0.5,FALSE,pearson,TRUE,ctx_w55_high,net_g05_w55_high,behavior,non_focal
g05,55,high,309,SEVERITY,SEVERITY,7,0.7528890,gamma_05,0.5,FALSE,pearson,TRUE,ctx_w55_high,net_g05_w55_high,belief,focal_belief
g05,55,low,376,AFF_FEAR,AFF_FEAR,4,1.0237396,gamma_05,0.5,FALSE,pearson,TRUE,ctx_w55_low,net_g05_w55_low,affect,non_focal


In [7]:
# ── Context-level summary ─────────────────────────────────────────────────────
edge_counts <- edges_all %>%
  mutate(
    is_bridge = (from %in% PSY_NODES & to %in% BEH_NODES) |
                (from %in% BEH_NODES & to %in% PSY_NODES)
  ) %>%
  group_by(spec, wave, trust_stratum, n_cc) %>%
  summarise(
    n_edges_nonzero = n(),
    n_bridge_edges  = sum(is_bridge),
    .groups = "drop"
  )

beh_agg <- node_metrics %>%
  filter(domain == "behavior") %>%
  group_by(spec, wave, trust_stratum, n_cc) %>%
  summarise(
    beh_degree_mean   = mean(degree),
    beh_degree_min    = min(degree),
    beh_strength_mean = mean(strength),
    .groups = "drop"
  )

context_summary <- edge_counts %>%
  left_join(beh_agg, by = c("spec","wave","trust_stratum","n_cc")) %>%
  left_join(GAMMA_LOOKUP, by = "spec") %>%
  mutate(
    context_id = paste0("ctx_w", wave, "_", trust_stratum),
    network_id = paste0("net_", spec, "_w", wave, "_", trust_stratum)
  ) %>%
  arrange(spec, wave, trust_stratum)

write_csv(context_summary, file.path(NETWORK_DIR, "context_summary.csv"))
cat("Saved: context_summary.csv —", nrow(context_summary), "rows\n")
context_summary

Saved: context_summary.csv — 24 rows


spec,wave,trust_stratum,n_cc,n_edges_nonzero,n_bridge_edges,beh_degree_mean,beh_degree_min,beh_strength_mean,spec_id,gamma_ebic,thresholded,cor_method,in_kg,context_id,network_id
<chr>,<int>,<chr>,<int>,<int>,<int>,<dbl>,<int>,<dbl>,<chr>,<dbl>,<lgl>,<chr>,<lgl>,<chr>,<chr>
g05,55,high,309,25,13,6.25,5,0.6982227,gamma_05,0.50,FALSE,pearson,TRUE,ctx_w55_high,net_g05_w55_high
g05,55,low,376,23,11,5.75,5,0.7857290,gamma_05,0.50,FALSE,pearson,TRUE,ctx_w55_low,net_g05_w55_low
g05,56,high,351,21,10,5.50,4,0.6916435,gamma_05,0.50,FALSE,pearson,TRUE,ctx_w56_high,net_g05_w56_high
g05,56,low,383,21,9,5.25,3,0.7482350,gamma_05,0.50,FALSE,pearson,TRUE,ctx_w56_low,net_g05_w56_low
g05,57,high,407,17,6,4.50,4,0.4964156,gamma_05,0.50,FALSE,pearson,TRUE,ctx_w57_high,net_g05_w57_high
g05,57,low,336,22,10,5.50,4,0.8297972,gamma_05,0.50,FALSE,pearson,TRUE,ctx_w57_low,net_g05_w57_low
g05_spearman,55,high,309,23,11,5.75,5,0.6277178,gamma_05_spearman,0.50,FALSE,spearman,FALSE,ctx_w55_high,net_g05_spearman_w55_high
g05_spearman,55,low,376,23,11,5.75,5,0.8337819,gamma_05_spearman,0.50,FALSE,spearman,FALSE,ctx_w55_low,net_g05_spearman_w55_low
g05_spearman,56,high,351,22,10,5.50,4,0.7252624,gamma_05_spearman,0.50,FALSE,spearman,FALSE,ctx_w56_high,net_g05_spearman_w56_high


In [8]:
# ── Edge-weight differences: high − low trust, within wave ───────────────────
# For each wave × spec pair, compute element-wise (W_high − W_low) on the
# upper-triangle of the adjacency matrix.
# Pairs where the difference is exactly zero (both networks agree) are retained
# with diff = 0; we filter ONLY at display time if needed, not at save time.
# This preserves complete information in the exported file.

adj_lookup <- adj_meta %>% select(spec, wave, trust_stratum, n_cc, path)

load_adj <- function(meta_row) {
  M <- as.matrix(read.csv(meta_row$path, row.names = 1, check.names = FALSE))
  M[NODE_VARS, NODE_VARS, drop = FALSE]
}

pairs <- adj_lookup %>%
  group_by(spec, wave) %>%
  summarise(
    has_low  = any(trust_stratum == "low"),
    has_high = any(trust_stratum == "high"),
    .groups  = "drop"
  ) %>%
  filter(has_low, has_high) %>%
  select(spec, wave)

diff_all <- map_dfr(seq_len(nrow(pairs)), function(i) {
  sp <- pairs$spec[i]
  wv <- pairs$wave[i]

  low_row  <- filter(adj_lookup, spec == sp, wave == wv, trust_stratum == "low")  %>% slice(1)
  high_row <- filter(adj_lookup, spec == sp, wave == wv, trust_stratum == "high") %>% slice(1)

  W_low  <- load_adj(low_row)
  W_high <- load_adj(high_row)

  D   <- W_high - W_low
  idx <- which(upper.tri(D), arr.ind = TRUE)

  tibble(
    spec                      = sp,
    wave                      = wv,
    from                      = NODE_VARS[idx[, 1]],
    to                        = NODE_VARS[idx[, 2]],
    diff_weight_high_minus_low = as.numeric(D[idx]),
    abs_diff                  = abs(as.numeric(D[idx])),
    n_low                     = low_row$n_cc,
    n_high                    = high_row$n_cc,
    from_domain               = NODE_DOMAIN[NODE_VARS[idx[, 1]]],
    to_domain                 = NODE_DOMAIN[NODE_VARS[idx[, 2]]],
    from_role                 = NODE_ROLE[NODE_VARS[idx[, 1]]],
    to_role                   = NODE_ROLE[NODE_VARS[idx[, 2]]]
  )
}) %>%
  left_join(GAMMA_LOOKUP, by = "spec") %>%
  mutate(comparison_id = paste0("diff_", spec, "_w", wave, "_high_minus_low")) %>%
  arrange(spec, wave, desc(abs_diff))

write_csv(diff_all, file.path(NETWORK_DIR, "edge_differences_high_minus_low.csv"))
cat("Saved: edge_differences_high_minus_low.csv —", nrow(diff_all), "rows\n")

# Show only non-zero differences for inspection
diff_all %>% filter(abs_diff > 0) %>% head(20)

Saved: edge_differences_high_minus_low.csv — 336 rows


spec,wave,from,to,diff_weight_high_minus_low,abs_diff,n_low,n_high,from_domain,to_domain,from_role,to_role,spec_id,gamma_ebic,thresholded,cor_method,in_kg,comparison_id
<chr>,<int>,<chr>,<chr>,<dbl>,<dbl>,<int>,<int>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<lgl>,<chr>,<lgl>,<chr>
g05,55,SEVERITY,USE2_AVOID,0.12576886,0.12576886,376,309,belief,behavior,focal_belief,non_focal,gamma_05,0.5,FALSE,pearson,TRUE,diff_g05_w55_high_minus_low
g05,55,SEVERITY,USE2_SPACE150,0.11373258,0.11373258,376,309,belief,behavior,focal_belief,non_focal,gamma_05,0.5,FALSE,pearson,TRUE,diff_g05_w55_high_minus_low
g05,55,USE2_MASK,USE2_SPACE150,-0.11344080,0.11344080,376,309,behavior,behavior,non_focal,non_focal,gamma_05,0.5,FALSE,pearson,TRUE,diff_g05_w55_high_minus_low
g05,55,WORRY_HEALTH_SYSTEM,USE2_HANDWASH20,-0.11261567,0.11261567,376,309,affect,behavior,non_focal,non_focal,gamma_05,0.5,FALSE,pearson,TRUE,diff_g05_w55_high_minus_low
g05,55,USE2_MASK,USE2_AVOID,-0.10658093,0.10658093,376,309,behavior,behavior,non_focal,non_focal,gamma_05,0.5,FALSE,pearson,TRUE,diff_g05_w55_high_minus_low
g05,55,SEVERITY,USE2_MASK,-0.09792896,0.09792896,376,309,belief,behavior,focal_belief,non_focal,gamma_05,0.5,FALSE,pearson,TRUE,diff_g05_w55_high_minus_low
g05,55,SEVERITY,WORRY_HEALTH_SYSTEM,-0.08422747,0.08422747,376,309,belief,affect,focal_belief,non_focal,gamma_05,0.5,FALSE,pearson,TRUE,diff_g05_w55_high_minus_low
g05,55,AFF_FEAR,WORRY_HEALTH_SYSTEM,0.08209579,0.08209579,376,309,affect,affect,non_focal,non_focal,gamma_05,0.5,FALSE,pearson,TRUE,diff_g05_w55_high_minus_low
g05,55,SEVERITY,AFF_FEAR,-0.07143053,0.07143053,376,309,belief,affect,focal_belief,non_focal,gamma_05,0.5,FALSE,pearson,TRUE,diff_g05_w55_high_minus_low


In [9]:
# ── Bridge-edge differences subset ───────────────────────────────────────────
bridge_diff <- diff_all %>%
  filter(
    (from_domain %in% c("belief", "affect") & to_domain == "behavior") |
    (from_domain == "behavior"              & to_domain %in% c("belief", "affect"))
  ) %>%
  arrange(spec, wave, desc(abs_diff))

write_csv(bridge_diff, file.path(NETWORK_DIR, "bridge_differences_high_minus_low.csv"))
cat("Saved: bridge_differences_high_minus_low.csv —", nrow(bridge_diff), "rows\n")
bridge_diff %>% filter(abs_diff > 0) %>% head(20)

Saved: bridge_differences_high_minus_low.csv — 192 rows


spec,wave,from,to,diff_weight_high_minus_low,abs_diff,n_low,n_high,from_domain,to_domain,from_role,to_role,spec_id,gamma_ebic,thresholded,cor_method,in_kg,comparison_id
<chr>,<int>,<chr>,<chr>,<dbl>,<dbl>,<int>,<int>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<lgl>,<chr>,<lgl>,<chr>
g05,55,SEVERITY,USE2_AVOID,0.125768865,0.125768865,376,309,belief,behavior,focal_belief,non_focal,gamma_05,0.5,FALSE,pearson,TRUE,diff_g05_w55_high_minus_low
g05,55,SEVERITY,USE2_SPACE150,0.113732578,0.113732578,376,309,belief,behavior,focal_belief,non_focal,gamma_05,0.5,FALSE,pearson,TRUE,diff_g05_w55_high_minus_low
g05,55,WORRY_HEALTH_SYSTEM,USE2_HANDWASH20,-0.112615672,0.112615672,376,309,affect,behavior,non_focal,non_focal,gamma_05,0.5,FALSE,pearson,TRUE,diff_g05_w55_high_minus_low
g05,55,SEVERITY,USE2_MASK,-0.097928962,0.097928962,376,309,belief,behavior,focal_belief,non_focal,gamma_05,0.5,FALSE,pearson,TRUE,diff_g05_w55_high_minus_low
g05,55,AFF_WORRY,USE2_SPACE150,-0.070541362,0.070541362,376,309,affect,behavior,non_focal,non_focal,gamma_05,0.5,FALSE,pearson,TRUE,diff_g05_w55_high_minus_low
g05,55,WORRY_HEALTH_SYSTEM,USE2_MASK,-0.060688631,0.060688631,376,309,affect,behavior,non_focal,non_focal,gamma_05,0.5,FALSE,pearson,TRUE,diff_g05_w55_high_minus_low
g05,55,AFF_FEAR,USE2_AVOID,-0.036241155,0.036241155,376,309,affect,behavior,non_focal,non_focal,gamma_05,0.5,FALSE,pearson,TRUE,diff_g05_w55_high_minus_low
g05,55,SEVERITY,USE2_HANDWASH20,0.034265297,0.034265297,376,309,belief,behavior,focal_belief,non_focal,gamma_05,0.5,FALSE,pearson,TRUE,diff_g05_w55_high_minus_low
g05,55,AFF_WORRY,USE2_HANDWASH20,0.031621419,0.031621419,376,309,affect,behavior,non_focal,non_focal,gamma_05,0.5,FALSE,pearson,TRUE,diff_g05_w55_high_minus_low


In [10]:
# ── Focal-belief bridges: SEVERITY ↔ behavior ────────────────────────────────
# Subset of bridge_df where one endpoint is the focal belief node.
focal_bridge <- bridge_df %>%
  filter(
    (from == FOCAL_NODE & to_domain   == "behavior") |
    (to   == FOCAL_NODE & from_domain == "behavior")
  ) %>%
  arrange(spec, wave, trust_stratum, desc(abs_weight))

write_csv(focal_bridge, file.path(NETWORK_DIR, "focal_belief_bridges.csv"))
cat("Saved: focal_belief_bridges.csv —", nrow(focal_bridge), "rows\n")
focal_bridge

Saved: focal_belief_bridges.csv — 47 rows


wave,trust_stratum,n_cc,spec,from,to,weight,spec_id,gamma_ebic,thresholded,⋯,context_id,network_id,edge_a,edge_b,edge_id,abs_weight,from_domain,to_domain,from_role,to_role
<int>,<chr>,<int>,<chr>,<chr>,<chr>,<dbl>,<chr>,<dbl>,<lgl>,⋯,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>
55,high,309,g05,SEVERITY,USE2_AVOID,0.140764339,gamma_05,0.50,FALSE,⋯,ctx_w55_high,net_g05_w55_high,SEVERITY,USE2_AVOID,net_g05_w55_high__SEVERITY__USE2_AVOID,0.140764339,belief,behavior,focal_belief,non_focal
55,high,309,g05,SEVERITY,USE2_SPACE150,0.127413906,gamma_05,0.50,FALSE,⋯,ctx_w55_high,net_g05_w55_high,SEVERITY,USE2_SPACE150,net_g05_w55_high__SEVERITY__USE2_SPACE150,0.127413906,belief,behavior,focal_belief,non_focal
55,high,309,g05,SEVERITY,USE2_MASK,-0.080601952,gamma_05,0.50,FALSE,⋯,ctx_w55_high,net_g05_w55_high,SEVERITY,USE2_MASK,net_g05_w55_high__SEVERITY__USE2_MASK,0.080601952,belief,behavior,focal_belief,non_focal
55,high,309,g05,SEVERITY,USE2_HANDWASH20,0.038896167,gamma_05,0.50,FALSE,⋯,ctx_w55_high,net_g05_w55_high,SEVERITY,USE2_HANDWASH20,net_g05_w55_high__SEVERITY__USE2_HANDWASH20,0.038896167,belief,behavior,focal_belief,non_focal
55,low,376,g05,SEVERITY,USE2_MASK,0.017327010,gamma_05,0.50,FALSE,⋯,ctx_w55_low,net_g05_w55_low,SEVERITY,USE2_MASK,net_g05_w55_low__SEVERITY__USE2_MASK,0.017327010,belief,behavior,focal_belief,non_focal
55,low,376,g05,SEVERITY,USE2_AVOID,0.014995474,gamma_05,0.50,FALSE,⋯,ctx_w55_low,net_g05_w55_low,SEVERITY,USE2_AVOID,net_g05_w55_low__SEVERITY__USE2_AVOID,0.014995474,belief,behavior,focal_belief,non_focal
55,low,376,g05,SEVERITY,USE2_SPACE150,0.013681328,gamma_05,0.50,FALSE,⋯,ctx_w55_low,net_g05_w55_low,SEVERITY,USE2_SPACE150,net_g05_w55_low__SEVERITY__USE2_SPACE150,0.013681328,belief,behavior,focal_belief,non_focal
55,low,376,g05,SEVERITY,USE2_HANDWASH20,0.004630871,gamma_05,0.50,FALSE,⋯,ctx_w55_low,net_g05_w55_low,SEVERITY,USE2_HANDWASH20,net_g05_w55_low__SEVERITY__USE2_HANDWASH20,0.004630871,belief,behavior,focal_belief,non_focal
56,high,351,g05,SEVERITY,USE2_AVOID,0.127075034,gamma_05,0.50,FALSE,⋯,ctx_w56_high,net_g05_w56_high,SEVERITY,USE2_AVOID,net_g05_w56_high__SEVERITY__USE2_AVOID,0.127075034,belief,behavior,focal_belief,non_focal


In [11]:
# ── Top-10 edges per network instance ────────────────────────────────────────
TOP_K <- 10L

top_edges <- edges_all %>%
  group_by(network_id) %>%
  arrange(desc(abs_weight), .by_group = TRUE) %>%
  slice_head(n = TOP_K) %>%
  ungroup() %>%
  arrange(network_id, desc(abs_weight))

write_csv(top_edges, file.path(NETWORK_DIR, paste0("top_edges_top", TOP_K, ".csv")))
cat("Saved: top_edges_top", TOP_K, ".csv —", nrow(top_edges), "rows\n")
head(top_edges, 10)

Saved: top_edges_top 10 .csv — 234 rows


wave,trust_stratum,n_cc,spec,from,to,weight,spec_id,gamma_ebic,thresholded,⋯,context_id,network_id,edge_a,edge_b,edge_id,abs_weight,from_domain,to_domain,from_role,to_role
<int>,<chr>,<int>,<chr>,<chr>,<chr>,<dbl>,<chr>,<dbl>,<lgl>,⋯,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>
55,high,309,g05_spearman,AFF_FEAR,AFF_WORRY,0.5399868,gamma_05_spearman,0.5,FALSE,⋯,ctx_w55_high,net_g05_spearman_w55_high,AFF_FEAR,AFF_WORRY,net_g05_spearman_w55_high__AFF_FEAR__AFF_WORRY,0.5399868,affect,affect,non_focal,non_focal
55,high,309,g05_spearman,USE2_AVOID,USE2_SPACE150,0.2593841,gamma_05_spearman,0.5,FALSE,⋯,ctx_w55_high,net_g05_spearman_w55_high,USE2_AVOID,USE2_SPACE150,net_g05_spearman_w55_high__USE2_AVOID__USE2_SPACE150,0.2593841,behavior,behavior,non_focal,non_focal
55,high,309,g05_spearman,USE2_HANDWASH20,USE2_SPACE150,0.2051245,gamma_05_spearman,0.5,FALSE,⋯,ctx_w55_high,net_g05_spearman_w55_high,USE2_HANDWASH20,USE2_SPACE150,net_g05_spearman_w55_high__USE2_HANDWASH20__USE2_SPACE150,0.2051245,behavior,behavior,non_focal,non_focal
55,high,309,g05_spearman,AFF_FEAR,SEVERITY,0.1897455,gamma_05_spearman,0.5,FALSE,⋯,ctx_w55_high,net_g05_spearman_w55_high,AFF_FEAR,SEVERITY,net_g05_spearman_w55_high__AFF_FEAR__SEVERITY,0.1897455,affect,belief,non_focal,focal_belief
55,high,309,g05_spearman,USE2_HANDWASH20,USE2_MASK,0.1832288,gamma_05_spearman,0.5,FALSE,⋯,ctx_w55_high,net_g05_spearman_w55_high,USE2_HANDWASH20,USE2_MASK,net_g05_spearman_w55_high__USE2_HANDWASH20__USE2_MASK,0.1832288,behavior,behavior,non_focal,non_focal
55,high,309,g05_spearman,USE2_MASK,USE2_SPACE150,0.1328180,gamma_05_spearman,0.5,FALSE,⋯,ctx_w55_high,net_g05_spearman_w55_high,USE2_MASK,USE2_SPACE150,net_g05_spearman_w55_high__USE2_MASK__USE2_SPACE150,0.1328180,behavior,behavior,non_focal,non_focal
55,high,309,g05_spearman,AFF_FEAR,WORRY_HEALTH_SYSTEM,0.1298846,gamma_05_spearman,0.5,FALSE,⋯,ctx_w55_high,net_g05_spearman_w55_high,AFF_FEAR,WORRY_HEALTH_SYSTEM,net_g05_spearman_w55_high__AFF_FEAR__WORRY_HEALTH_SYSTEM,0.1298846,affect,affect,non_focal,non_focal
55,high,309,g05_spearman,AFF_WORRY,WORRY_HEALTH_SYSTEM,0.1290801,gamma_05_spearman,0.5,FALSE,⋯,ctx_w55_high,net_g05_spearman_w55_high,AFF_WORRY,WORRY_HEALTH_SYSTEM,net_g05_spearman_w55_high__AFF_WORRY__WORRY_HEALTH_SYSTEM,0.1290801,affect,affect,non_focal,non_focal
55,high,309,g05_spearman,SEVERITY,USE2_SPACE150,0.1186974,gamma_05_spearman,0.5,FALSE,⋯,ctx_w55_high,net_g05_spearman_w55_high,SEVERITY,USE2_SPACE150,net_g05_spearman_w55_high__SEVERITY__USE2_SPACE150,0.1186974,belief,behavior,focal_belief,non_focal


In [12]:
# ── Coverage check: contexts present ─────────────────────────────────────────
contexts_found <- edges_all %>%
  distinct(spec, wave, trust_stratum, n_cc) %>%
  left_join(GAMMA_LOOKUP, by = "spec") %>%
  mutate(
    context_id = paste0("ctx_w",  wave, "_", trust_stratum),
    network_id = paste0("net_", spec, "_w", wave, "_", trust_stratum)
  ) %>%
  arrange(spec, wave, trust_stratum)

cat("Contexts in edges_all (spec × wave × trust):", nrow(contexts_found), "\n")
contexts_found

Contexts in edges_all (spec × wave × trust): 24 


spec,wave,trust_stratum,n_cc,spec_id,gamma_ebic,thresholded,cor_method,in_kg,context_id,network_id
<chr>,<int>,<chr>,<int>,<chr>,<dbl>,<lgl>,<chr>,<lgl>,<chr>,<chr>
g05,55,high,309,gamma_05,0.50,FALSE,pearson,TRUE,ctx_w55_high,net_g05_w55_high
g05,55,low,376,gamma_05,0.50,FALSE,pearson,TRUE,ctx_w55_low,net_g05_w55_low
g05,56,high,351,gamma_05,0.50,FALSE,pearson,TRUE,ctx_w56_high,net_g05_w56_high
g05,56,low,383,gamma_05,0.50,FALSE,pearson,TRUE,ctx_w56_low,net_g05_w56_low
g05,57,high,407,gamma_05,0.50,FALSE,pearson,TRUE,ctx_w57_high,net_g05_w57_high
g05,57,low,336,gamma_05,0.50,FALSE,pearson,TRUE,ctx_w57_low,net_g05_w57_low
g05_spearman,55,high,309,gamma_05_spearman,0.50,FALSE,spearman,FALSE,ctx_w55_high,net_g05_spearman_w55_high
g05_spearman,55,low,376,gamma_05_spearman,0.50,FALSE,spearman,FALSE,ctx_w55_low,net_g05_spearman_w55_low
g05_spearman,56,high,351,gamma_05_spearman,0.50,FALSE,spearman,FALSE,ctx_w56_high,net_g05_spearman_w56_high


## Bootstrap uncertainty — edge-level stability metrics

Nonparametric bootstrap distributions (stored in `.rds` files) are summarised
to obtain per-edge stability heuristics: bootstrap mean, SD, 95 % percentile CI,
and proportion of bootstrap samples in which the edge is nonzero.

Bootstrap files exist for `g05` and `g075` only.  Edges from `g05_thr` will
have `NA` bootstrap columns in `edge_metrics.csv`, which is correct by design.

**Note on `summarise_boot_object`:** `bootnet` stores results in `bootTable`
(one row per bootstrap × edge) and `sampleTable` (sample estimates).  Column
names differ across `bootnet` versions, so the helper uses a defensive lookup.

In [13]:
# ── Bootstrap edge-stability summary ─────────────────────────────────────────

# Helper: robustly find column name from a set of candidates
find_col <- function(df, candidates) {
  hit <- intersect(candidates, names(df))
  if (length(hit) == 0)
    stop("None of the expected columns found. Available: ",
         paste(names(df), collapse=", "))
  hit[1]
}

summarise_boot_object <- function(path, spec, wave, trust_stratum, n_cc) {
  obj <- readRDS(path)

  bt <- obj$bootTable
  st <- obj$sampleTable
  if (is.null(bt) || is.null(st))
    stop("bootTable or sampleTable missing in: ", basename(path))

  # ── Resolve column names defensively (bootnet version-safe) ────────────────
  bt_from <- find_col(bt, c("node1","from","Var1"))
  bt_to   <- find_col(bt, c("node2","to",  "Var2"))
  bt_val  <- find_col(bt, c("value","Value","stat"))
  bt_type <- intersect(c("type","Type"), names(bt))

  st_from <- find_col(st, c("node1","from","Var1"))
  st_to   <- find_col(st, c("node2","to",  "Var2"))
  st_val  <- find_col(st, c("value","Value","stat"))
  st_type <- intersect(c("type","Type"), names(st))

  # Keep only "edge" rows if a type column exists
  if (length(bt_type) > 0) bt <- bt[bt[[bt_type[1]]] == "edge", , drop=FALSE]
  if (length(st_type) > 0) st <- st[st[[st_type[1]]] == "edge", , drop=FALSE]

  # ── Canonical edge keys (alphabetic min/max) ───────────────────────────────
  bf  <- as.character(bt[[bt_from]]);  bt2 <- as.character(bt[[bt_to]])
  bv  <- as.numeric(bt[[bt_val]])
  k1  <- pmin(bf, bt2);               k2  <- pmax(bf, bt2)
  bt_df <- tibble(edge_a = k1, edge_b = k2, value = bv)

  sf  <- as.character(st[[st_from]]);  st2 <- as.character(st[[st_to]])
  sv  <- as.numeric(st[[st_val]])
  sk1 <- pmin(sf, st2);               sk2 <- pmax(sf, st2)
  sample_tbl <- tibble(edge_a = sk1, edge_b = sk2, sample_weight = sv) %>%
    group_by(edge_a, edge_b) %>%
    summarise(sample_weight = mean(sample_weight, na.rm = TRUE), .groups = "drop")

  # ── Summarise bootstrap distribution per edge ─────────────────────────────
  bt_df %>%
    group_by(edge_a, edge_b) %>%
    summarise(
      boot_mean    = mean(value, na.rm = TRUE),
      boot_sd      = sd(value,   na.rm = TRUE),
      ci_low       = as.numeric(quantile(value, 0.025, na.rm = TRUE)),
      ci_high      = as.numeric(quantile(value, 0.975, na.rm = TRUE)),
      prop_nonzero = mean(value != 0, na.rm = TRUE),
      n_boot       = sum(!is.na(value)),
      .groups      = "drop"
    ) %>%
    left_join(sample_tbl, by = c("edge_a","edge_b")) %>%
    mutate(
      spec          = .env$spec,
      wave          = .env$wave,
      trust_stratum = .env$trust_stratum,
      n_cc          = .env$n_cc
    ) %>%
    left_join(GAMMA_LOOKUP %>% select(spec, spec_id, gamma_ebic), by = "spec") %>%
    mutate(
      context_id = paste0("ctx_w", wave, "_", trust_stratum),
      network_id = paste0("net_", spec, "_w", wave, "_", trust_stratum),
      edge_id    = paste(network_id, edge_a, edge_b, sep = "__"),
      from       = edge_a,
      to         = edge_b
    ) %>%
    select(
      spec, spec_id, gamma_ebic, wave, trust_stratum, n_cc,
      context_id, network_id, edge_id,
      from, to, edge_a, edge_b, sample_weight,
      boot_mean, boot_sd, ci_low, ci_high, prop_nonzero, n_boot
    )
}

if (nrow(boot_meta) == 0) {
  warning("No bootstrap RDS files found. edge_metrics.csv will have NA bootstrap columns.")
  boot_edge_metrics <- tibble(
    spec=character(), spec_id=character(), gamma_ebic=numeric(),
    wave=integer(), trust_stratum=character(), n_cc=integer(),
    context_id=character(), network_id=character(), edge_id=character(),
    from=character(), to=character(), edge_a=character(), edge_b=character(),
    sample_weight=numeric(), boot_mean=numeric(), boot_sd=numeric(),
    ci_low=numeric(), ci_high=numeric(), prop_nonzero=numeric(), n_boot=integer()
  )
} else {
  boot_edge_metrics <- pmap_dfr(
    list(
      path          = boot_meta$path,
      spec          = boot_meta$spec,
      wave          = boot_meta$wave,
      trust_stratum = boot_meta$trust_stratum,
      n_cc          = boot_meta$n_cc
    ),
    summarise_boot_object
  ) %>%
    arrange(spec, wave, trust_stratum, desc(abs(sample_weight)))
}

write_csv(boot_edge_metrics, file.path(NETWORK_DIR, "edge_bootstrap_metrics.csv"))
cat("Saved: edge_bootstrap_metrics.csv —", nrow(boot_edge_metrics), "rows\n")

Saved: edge_bootstrap_metrics.csv — 336 rows


In [14]:
# ── Merge point estimates with bootstrap summaries → edge_metrics.csv ─────────
# g05_thr edges have no matching bootstrap row; their bootstrap columns are NA.
# This is expected and correct — NA boot columns are a feature, not a defect.

boot_join_cols <- c("spec","wave","trust_stratum","n_cc","edge_a","edge_b")
boot_payload   <- c("sample_weight","boot_mean","boot_sd",
                    "ci_low","ci_high","prop_nonzero","n_boot")

edge_metrics <- edges_all %>%
  left_join(
    boot_edge_metrics %>% select(all_of(c(boot_join_cols, boot_payload))),
    by = boot_join_cols
  ) %>%
  arrange(spec, wave, trust_stratum, desc(abs_weight))

# Report missing-rate by spec (g05_thr should be 100%; g05/g075 should be 0%)
miss_by_spec <- edge_metrics %>%
  group_by(spec) %>%
  summarise(pct_na_boot_mean = mean(is.na(boot_mean)) * 100, .groups = "drop")

cat("Bootstrap NA rate by spec (expected 100% for g05_thr, ~0% for g05/g075):\n")
print(miss_by_spec)

write_csv(edge_metrics, file.path(NETWORK_DIR, "edge_metrics.csv"))
cat("Saved: edge_metrics.csv —", nrow(edge_metrics), "rows\n")
head(edge_metrics, 10)

Bootstrap NA rate by spec (expected 100% for g05_thr, ~0% for g05/g075):


# A tibble: 4 × 2
  spec         pct_na_boot_mean
  <chr>                   <dbl>
1 g05                         0
2 g05_spearman              100
3 g05_thr                   100
4 g075                        0
Saved: edge_metrics.csv — 442 rows


wave,trust_stratum,n_cc,spec,from,to,weight,spec_id,gamma_ebic,thresholded,⋯,to_domain,from_role,to_role,sample_weight,boot_mean,boot_sd,ci_low,ci_high,prop_nonzero,n_boot
<int>,<chr>,<int>,<chr>,<chr>,<chr>,<dbl>,<chr>,<dbl>,<lgl>,⋯,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>
55,high,309,g05,AFF_FEAR,AFF_WORRY,0.5773954,gamma_05,0.5,FALSE,⋯,affect,non_focal,non_focal,0.5773954,0.5601251,0.05011639,0.45658259,0.6468484,1.000,1000
55,high,309,g05,USE2_AVOID,USE2_SPACE150,0.2379412,gamma_05,0.5,FALSE,⋯,behavior,non_focal,non_focal,0.2379412,0.2329179,0.05296988,0.12388081,0.3361304,1.000,1000
55,high,309,g05,USE2_MASK,USE2_SPACE150,0.2135914,gamma_05,0.5,FALSE,⋯,behavior,non_focal,non_focal,0.2135914,0.2007787,0.06549991,0.06635812,0.3198839,0.997,1000
55,high,309,g05,USE2_HANDWASH20,USE2_SPACE150,0.2033636,gamma_05,0.5,FALSE,⋯,behavior,non_focal,non_focal,0.2033636,0.1971649,0.06076647,0.07526566,0.3161542,1.000,1000
55,high,309,g05,AFF_FEAR,SEVERITY,0.1963276,gamma_05,0.5,FALSE,⋯,belief,non_focal,focal_belief,0.1963276,0.1991228,0.04964673,0.09253927,0.2916246,1.000,1000
55,high,309,g05,USE2_HANDWASH20,USE2_MASK,0.1832615,gamma_05,0.5,FALSE,⋯,behavior,non_focal,non_focal,0.1832615,0.1705988,0.05713821,0.05465144,0.2838080,0.995,1000
55,high,309,g05,AFF_FEAR,WORRY_HEALTH_SYSTEM,0.1779615,gamma_05,0.5,FALSE,⋯,affect,non_focal,non_focal,0.1779615,0.1737192,0.05830889,0.05372568,0.2805723,0.999,1000
55,high,309,g05,USE2_AVOID,USE2_HANDWASH20,0.1445860,gamma_05,0.5,FALSE,⋯,behavior,non_focal,non_focal,0.1445860,0.1396358,0.05290904,0.03149183,0.2422949,0.995,1000
55,high,309,g05,SEVERITY,USE2_AVOID,0.1407643,gamma_05,0.5,FALSE,⋯,behavior,focal_belief,non_focal,0.1407643,0.1311757,0.05212756,0.02478151,0.2346041,0.989,1000


## KG-ready metadata tables

`kg_contexts.csv` and `kg_network_instances.csv` are the entity/node tables
loaded into the Belief Knowledge Graph (BKG). Estimation metadata (gamma,
cor_method) is stored as properties on NetworkInstance.

**Scope:** `g05` only — 6 NetworkInstance nodes
(3 waves × 2 trust strata).

**`global_strength`** (sum of absolute edge weights) is computed here and attached
to each NetworkInstance row.


In [15]:
# ── KG-ready metadata tables ──────────────────────────────────────────────────
# Only the main specification (g05) is stored in the BKG.
# GammaSpec is no longer a separate node type; estimation metadata (gamma,
# cor_method) is stored as properties on NetworkInstance.

# global_strength: sum of absolute edge weights per network instance (KG specs only)
gs_per_network <- edges_all %>%
  filter(spec %in% KG_SPECS$spec) %>%
  group_by(network_id) %>%
  summarise(global_strength = sum(abs_weight), .groups = "drop")

kg_network_instances <- edge_metrics %>%
  filter(spec %in% KG_SPECS$spec) %>%
  distinct(spec, wave, trust_stratum, n_cc) %>%
  left_join(KG_SPECS %>% select(spec, spec_id, gamma_ebic, cor_method), by = "spec") %>%
  mutate(
    context_id = paste0("ctx_w", wave, "_", trust_stratum),
    network_id = paste0("net_", spec, "_w", wave, "_", trust_stratum),
    gamma      = gamma_ebic,
    cor_method = cor_method
  ) %>%
  left_join(gs_per_network, by = "network_id") %>%
  arrange(spec, wave, trust_stratum)

kg_contexts <- kg_network_instances %>%
  distinct(context_id, wave, trust_stratum, n_cc) %>%
  mutate(
    trust_variable          = TRUST_VARIABLE,
    trust_grouping_rule     = TRUST_GROUPING_RULE,
    middle_tercile_excluded = MIDDLE_TERCILE_EXCLUDED
  ) %>%
  arrange(wave, trust_stratum)

# ── Assertions ────────────────────────────────────────────────────────────────
n_instances <- nrow(kg_network_instances)
n_contexts  <- nrow(kg_contexts)

# With GammaSpec removed, we expect exactly one instance per context
if (n_instances != n_contexts)
  stop(
    "Expected ", n_contexts, " KG instances (one per context) ",
    "but found ", n_instances, "."
  )

n_gs_na <- sum(is.na(kg_network_instances$global_strength))
if (n_gs_na > 0)
  stop(n_gs_na, " network instance(s) have NA global_strength. ",
       "Check that edges_all covers all KG network_ids.")

cat("KG network instances  :", n_instances, "(one per context)\n")
cat("Contexts              :", n_contexts, "\n")
cat("global_strength range : [",
    round(min(kg_network_instances$global_strength), 4), ",",
    round(max(kg_network_instances$global_strength), 4), "]\n")

write_csv(kg_contexts,            file.path(NETWORK_DIR, "kg_contexts.csv"))
write_csv(kg_network_instances,   file.path(NETWORK_DIR, "kg_network_instances.csv"))

cat("Saved: kg_contexts.csv, kg_network_instances.csv\n")
cat("Note: kg_gamma_specs.csv is no longer exported (GammaSpec removed from BKG schema).\n")
kg_network_instances


KG network instances  : 6 (one per context)
Contexts              : 6 
global_strength range : [ 2.3034 , 3.265 ]
Saved: kg_contexts.csv, kg_network_instances.csv
Note: kg_gamma_specs.csv is no longer exported (GammaSpec removed from BKG schema).


spec,wave,trust_stratum,n_cc,spec_id,gamma_ebic,cor_method,context_id,network_id,gamma,global_strength
<chr>,<int>,<chr>,<int>,<chr>,<dbl>,<chr>,<chr>,<chr>,<dbl>,<dbl>
g05,55,high,309,gamma_05,0.5,pearson,ctx_w55_high,net_g05_w55_high,0.5,2.948462
g05,55,low,376,gamma_05,0.5,pearson,ctx_w55_low,net_g05_w55_low,0.5,3.177500
g05,56,high,351,gamma_05,0.5,pearson,ctx_w56_high,net_g05_w56_high,0.5,2.655842
g05,56,low,383,gamma_05,0.5,pearson,ctx_w56_low,net_g05_w56_low,0.5,3.048761
g05,57,high,407,gamma_05,0.5,pearson,ctx_w57_high,net_g05_w57_high,0.5,2.303442
g05,57,low,336,gamma_05,0.5,pearson,ctx_w57_low,net_g05_w57_low,0.5,3.265030
